# Lab 5 - EDA with SQL

In [1]:
import pandas as pd, sqlite3
con = sqlite3.connect('my_data1.db')
df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv')
df.to_sql('SPACEXTBL', con, if_exists='replace', index=False)
cur = con.cursor()
cur.execute('DROP TABLE IF EXISTS SPACEXTABLE')
cur.execute('CREATE TABLE SPACEXTABLE AS SELECT * FROM SPACEXTBL WHERE Date IS NOT NULL')
def q(sql):
    return pd.read_sql_query(sql, con)
print('loaded')

loaded


### 1. Unique launch sites

In [2]:
q('SELECT DISTINCT Launch_Site FROM SPACEXTABLE')

,Launch_Site
0,CCAFS LC-40
1,VAFB SLC-4E
2,KSC LC-39A
3,CCAFS SLC-40


### 2. 5 records where launch sites begin with 'CCA'

In [3]:
q("SELECT * FROM SPACEXTABLE WHERE Launch_Site LIKE 'CCA%' LIMIT 5")

,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


### 3. Total payload mass carried by boosters launched by NASA (CRS)

In [4]:
q("SELECT SUM(PAYLOAD_MASS__KG_) AS Total FROM SPACEXTABLE WHERE Customer='NASA (CRS)'")

,Total
0,45596


### 4. Average payload mass carried by booster version F9 v1.1

In [5]:
q("SELECT AVG(PAYLOAD_MASS__KG_) AS Average FROM SPACEXTABLE WHERE Booster_Version='F9 v1.1'")

,Average
0,2928.4


### 5. Date of the first successful landing on a ground pad

In [6]:
q("SELECT MIN(Date) AS First FROM SPACEXTABLE WHERE Landing_Outcome='Success (ground pad)'")

,First
0,2015-12-22


### 6. Boosters with success on drone ship and payload 4000-6000 kg

In [7]:
q("SELECT DISTINCT Booster_Version FROM SPACEXTABLE WHERE Landing_Outcome='Success (drone ship)' AND PAYLOAD_MASS__KG_ BETWEEN 4000 AND 6000")

,Booster_Version
0,F9 FT B1022
1,F9 FT B1026
2,F9 FT B1021.2
3,F9 FT B1031.2


### 7. Total number of successful and failure mission outcomes

In [8]:
q('SELECT Mission_Outcome, COUNT(*) AS Total FROM SPACEXTABLE GROUP BY Mission_Outcome')

,Mission_Outcome,Total
0,Failure (in flight),1
1,Success,98
2,Success,1
3,Success (payload status unclear),1


### 8. Booster versions that carried the maximum payload

In [9]:
q('SELECT DISTINCT Booster_Version FROM SPACEXTABLE WHERE PAYLOAD_MASS__KG_=(SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTABLE)')

,Booster_Version
0,F9 B5 B1048.4
1,F9 B5 B1049.4
2,F9 B5 B1051.3
3,F9 B5 B1056.4
4,F9 B5 B1048.5
5,F9 B5 B1051.4
6,F9 B5 B1049.5
7,F9 B5 B1060.2
8,F9 B5 B1058.3
9,F9 B5 B1051.6


### 9. 2015 failed drone-ship landings with month numbers

In [10]:
q("SELECT substr(Date,6,2) AS Month, Booster_Version, Launch_Site FROM SPACEXTABLE WHERE Landing_Outcome='Failure (drone ship)' AND substr(Date,1,4)='2015'")

,Month,Booster_Version,Launch_Site
0,01,F9 v1.1 B1012,CCAFS LC-40
1,04,F9 v1.1 B1015,CCAFS LC-40


### 10. Rank landing outcomes between 2010-06-04 and 2017-03-20

In [11]:
q("SELECT Landing_Outcome, COUNT(*) AS Count FROM SPACEXTABLE WHERE Date BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY Landing_Outcome ORDER BY Count DESC")

,Landing_Outcome,Count
0,No attempt,10
1,Success (drone ship),5
2,Failure (drone ship),5
3,Success (ground pad),3
4,Controlled (ocean),3
5,Uncontrolled (ocean),2
6,Failure (parachute),2
7,Precluded (drone ship),1
